# 🍹 Sales Prediction Dataset Documentation

## 0. Quick Summary

### Inputs (Features)
- **DayType**  
- **Weather_8am_1pm**  
- **Weather_2pm_7pm**  
- **Weather_8pm_11pm**  
- **Traffic_Level**  
- **SchoolHoliday**  
- **PublicHoliday**  
- **Festival**  
- **BackToSchool**  
- **ExamSeason**  
- **PayDay**  
- **BigEventNearby**  
- **Promo_5PercentDiscount**  
- **Promo_10PercentDiscount**  
- **Promo_15PercentDiscount**  
- **Promo_20PercentDiscount**  
- **Promo_Buy1Free1**  
- **Promo_Buy2Free1**  
- **Promo_NewProductLaunch**  
- **Promo_AnniversarySale**  
- **Promo_SocialMediaCampaign**

### Outputs (Targets)
- **Drink_IcedCaramelMacchiato**  
- **Drink_MochaFrappe**  
- **Drink_StrawberrySmoothie**  
- **Drink_MangoPassionfruitCooler**  
- **Drink_MatchaLatte**  
- **Drink_ChocolateHazelnutShake**  
- **Drink_BlueberryLemonadeSparkler**  
- **Drink_AvocadoCoffee**

*(All targets are integer counts = cups sold)*

---

## 1. Data Collection & Storage

Each record should represent **one day** (or per shift/hour if higher frequency is needed).

### Columns

- **Date (ISO format)**  
- **Day (e.g., Monday, Tuesday...)**  
- **DayType** → Weekday / Weekend  
- **Weather_8am_1pm, Weather_2pm_7pm, Weather_8pm_11pm**  
  - *Categorical: Sunny / Partly Cloudy / Cloudy / Light Rain / Heavy Rain / Thunderstorm*  
- **Traffic_Level** → Low / Medium / High  

### Binary Flags (0/1)
- **SchoolHoliday**  
- **PublicHoliday**  
- **Festival**  
- **BackToSchool**  
- **ExamSeason**  
- **PayDay**  
- **BigEventNearby**

### Promo Flags (0/1)
- **Promo_5PercentDiscount**  
- **Promo_10PercentDiscount**  
- **Promo_15PercentDiscount**  
- **Promo_20PercentDiscount**  
- **Promo_Buy1Free1**  
- **Promo_Buy2Free1**  
- **Promo_NewProductLaunch**  
- **Promo_AnniversarySale**  
- **Promo_SocialMediaCampaign**

### Targets (Cups Sold, Integers)
- **Drink_IcedCaramelMacchiato**  
- **Drink_MochaFrappe**  
- **Drink_StrawberrySmoothie**  
- **Drink_MangoPassionfruitCooler**  
- **Drink_MatchaLatte**  
- **Drink_ChocolateHazelnutShake**  
- **Drink_BlueberryLemonadeSparkler**  
- **Drink_AvocadoCoffee**

### Optional
- **Total_Sales** → Auto (sum of all drinks)  
- **Notes** → Free text (events, special observations, anomalies, etc.)  


### Data Preperation

In [ ]:
import pandas as pd

sales_df = pd.read_excel("./dataset/sales.xlsx", "sale1")

feature_cols = [
    "DayType", "Weather_8am_1pm", "Weather_2pm_7pm", "Weather_8pm_11pm",
    "Traffic_Level", "SchoolHoliday", "PublicHoliday", "Festival",
    "BackToSchool", "ExamSeason", "PayDay", "BigEventNearby",
    "Promo_5PercentDiscount", "Promo_10PercentDiscount",
    "Promo_15PercentDiscount", "Promo_20PercentDiscount",
    "Promo_Buy1Free1", "Promo_Buy2Free1", "Promo_NewProductLaunch",
    "Promo_AnniversarySale", "Promo_SocialMediaCampaign"
]

# Outputs (the 8 drinks)
target_cols = [
    "Drink_IcedCaramelMacchiato", "Drink_MochaFrappe",
    "Drink_StrawberrySmoothie", "Drink_MangoPassionfruitCooler",
    "Drink_MatchaLatte", "Drink_ChocolateHazelnutShake",
    "Drink_BlueberryLemonadeSparkler", "Drink_AvocadoCoffee"
]

X = sales_df[feature_cols]
y = sales_df[target_cols]

print("\n📊 Dataset Overview:\n")
print(sales_df.head())          # preview first rows
print(sales_df.info())          # check column types
print(sales_df.describe())      # summary stats for numeric cols

In [ ]:
# Encode the label using label encoding(sbb sikit)
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()


# Split the data
from sklearn.model_selection import train_test_split 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)


### Preprocessing

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols) #handle_unknown="ignore" jika kalau test set ada category baru yang tak ada dalam train, model tak crash
    ]
)

#Apply Preprocessing
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train shape after preprocessing: \n", X_train_processed.shape)
print("X_test shape after preprocessing: \n", X_test_processed.shape)


In [ ]:
from catboost import CatBoostRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Define CatBoost model
catboost_model = CatBoostRegressor(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=0,
    random_state=42
)

# Wrap in MultiOutputRegressor for multi-target
multi_model = MultiOutputRegressor(catboost_model)

# Train
multi_model.fit(X_train, y_train)

# Predict
y_pred = multi_model.predict(X_test)

# Evaluate
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred, multioutput="uniform_average")

print("📊 CatBoost Multi-Output Performance")
print(f"Mean Absolute Error: {mae:.2f}")
print(f"R² Score: {r2:.2f}")
